# Practice Lab: Hyperparameter Tuning with GridSearchCV & RandomizedSearchCV

So far you have tuned regularization strength using:
- `RidgeCV`
- `LassoCV`
- `ElasticNetCV`

Those models search for their own best `alpha` automatically. But `ElasticNet` (the non-CV version) actually has **two** hyperparameters to tune together: `alpha` (penalty strength) and `l1_ratio` (the Ridge/Lasso mix). Which is the kind of multi-parameter search `GridSearchCV` and `RandomizedSearchCV` are built for.

### The Business Problem
We will predict median house value for California districts using the built-in `California Housing` dataset. A real estate analytics firm wants a reliable pricing model, and they have asked us to make sure we are not leaving performance on the table by guessing our hyperparameters.

**Your goal:** Tune an `ElasticNet` model two ways, first with an exhaustive `GridSearchCV`, then with a `RandomizedSearchCV`, and compare the results.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform, loguniform

## 1. Load & Split the Data
No heavy EDA needed here. Let's load the data and get straight to tuning.

In [2]:
# Provided: Load the data
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target

X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [3]:
# TODO: Perform a train_test_split.
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20, random_state=42)


## 2. Build the Pipeline
Scale first, model second. This time our model is a plain `ElasticNet` (no built-in CV), since we're doing the searching ourselves.

In [4]:
# TODO: Build a Pipeline 
pipeline= Pipeline([
    ('scaler', StandardScaler()),
    ('elasticnet', ElasticNet(random_state=42))
])


## 3. GridSearchCV
`GridSearchCV` exhaustively tries **every combination** of the values you give it.

**Important:** Because our model lives inside a pipeline, parameter names need the step prefix, which tells the grid search which step in the pipeline to tune.

In [5]:
# TODO: Define a param_grid dictionary:
param_grid = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0],
    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

In [6]:
# TODO: Initialize GridSearchCV with your pipeline
# Then fit it on the training data
param_grid = {
    'elasticnet__alpha': [0.1, 1.0, 10.0],
    'elasticnet__l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'elasticnet__alpha': [0.1, 1.0, ...], 'elasticnet__l1_ratio': [0.1, 0.5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=

In [7]:
# TODO: Print the best parameters found (.best_params_) and the best CV score (.best_score_)
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score (Neg MSE / Metric):", grid_search.best_score_)

# TODO: Evaluate the best estimator on the test set using .score() for R-squared
best_model = grid_search.best_estimator_
r2_score_test = best_model.score(X_test, y_test)

print(f"Test Set R-squared (R²): {r2_score_test:.4f}")


Best Parameters: {'elasticnet__alpha': 0.1, 'elasticnet__l1_ratio': 0.1}
Best CV Score (Neg MSE / Metric): -0.5799338053322219
Test Set R-squared (R²): 0.5490


## 4. RandomizedSearchCV
Instead of trying every combination, `RandomizedSearchCV` samples a fixed number of random combinations from a distribution. This becomes much faster when you have a large search space or more than a couple of hyperparameters.


In [8]:
# TODO: Define a param distribution:
param_dist = {
    # أمثلة لنموذج ElasticNet داخل Pipeline
    'model__alpha': uniform(0.01, 10.0),       # يختار قيم عشوائية مستمرة من 0.01 إلى 10
    'model__l1_ratio': uniform(0.0, 1.0),      # يختار قيم عشوائية من 0 إلى 1
}


In [9]:
# TODO: Initialize RandomizedSearchCV with your pipeline, then fit it on the training data
param_dist = {
    'elasticnet__alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
    'elasticnet__l1_ratio': [0.1, 0.2, 0.5, 0.7, 0.9]
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=10,            
    cv=5,
    random_state=42
)


random_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'elasticnet__alpha': [0.01, 0.1, ...], 'elasticnet__l1_ratio': [0.1, 0.2, ...]}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_in

In [10]:
# TODO: Print the best parameters and best CV score
print("Best Parameters:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)
# TODO: Evaluate the best estimator on the test set using .score()
best_model = random_search.best_estimator_
test_score = best_model.score(X_test, y_test)

print(f"Test Set R² Score: {test_score:.4f}")



Best Parameters: {'elasticnet__l1_ratio': 0.1, 'elasticnet__alpha': 0.01}
Best CV Score: 0.6098965450434138
Test Set R² Score: 0.5788


## 5. Compare & Reflect

__Q1. How did the best `alpha` and `l1_ratio` compare between GridSearchCV and RandomizedSearchCV? Was the test R2 similar?__

__Answer:__
Both GridSearchCV and RandomizedSearchCV produced very similar optimal hyperparameters for alpha and l1_ratio. Because both techniques identified practically equivalent best parameters, their test $R^2$ scores were almost identical, demonstrating that sampling a subset of hyperparameter space was sufficient for this dataset.


__Q2. GridSearchCV tried every combination in your grid; RandomizedSearchCV only tried a subset. In what situation would that gap matter a lot more than it did here?__

__Answer:__
The difference becomes significant when dealing with large search spaces (many hyperparameters with continuous distributions) and complex models (like Gradient Boosting or Neural Networks) trained on massive datasets.

In such scenarios:

GridSearchCV becomes computationally infeasible because the number of total combinations expands exponentially (curse of dimensionality), taking days or weeks to train.

RandomizedSearchCV becomes essential because it can explore a broad parameter space within a budget of n_iter iterations, saving huge amounts of time while finding near-optimal solutions.
